# Minimal Statistical Test

Only the essentials:
1. Set `DATASETS` (two files)
2. Set `METRIC_KEY`
3. Run the notebook


In [9]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from TSB_AD.evaluation.metrics import get_metrics # type: ignore
from TSB_AD.model_wrapper import run_Unsupervise_AD # type: ignore
from TSB_AD.models.PCA import PCA

warnings.filterwarnings("ignore")

## Configuration (Edit This Cell)

In [10]:
METHOD_NAME = "Sub_PCA"

DATASETS = [
    Path("../datasets/TSB-AD-U/001_NAB_id_1_Facility_tr_1007_1st_2014.csv"),
    Path("../datasets/TSB-AD-U/002_NAB_id_2_WebService_tr_1500_1st_4106.csv"),
]

METRIC_KEY = "AUC-ROC"  # Example alternatives: "AUC-PR", "PA-F1", "VUS-PR"

print(f"Method: {METHOD_NAME}")
print(f"Datasets: {[p.name for p in DATASETS]}")
print(f"Metric: {METRIC_KEY}")

Method: Sub_PCA
Datasets: ['001_NAB_id_1_Facility_tr_1007_1st_2014.csv', '002_NAB_id_2_WebService_tr_1500_1st_4106.csv']
Metric: AUC-ROC


In [11]:
rows = []

# for dataset_path in DATASETS:
dataset_path = DATASETS[0]
df = pd.read_csv(dataset_path).dropna()
data = df.iloc[:, :-1].values.astype(float)
labels = df["Label"].astype(int).to_numpy()

# if return is str -> error
scores = run_Unsupervise_AD(METHOD_NAME, data, periodicity=1)
if isinstance(scores, str):
    raise RuntimeError(scores)

print(f"Shape of scores: {scores.shape}, Shape of labels: {labels.shape}")
# ravel -> to 1D array (important when AD on multivariate data)
scores = np.asarray(scores)
print(f"Shape of scores after conversion: {scores.shape}")
scores = scores.ravel()
print(f"Shape of scores after ravel: {scores.shape}")
# Ensure scores and labels have the same length
n = min(len(scores), len(labels))
scores = scores[:n]
labels = labels[:n]

metrics = get_metrics(scores, labels)
# Check if my selected metric is in the returned metrics
if METRIC_KEY not in metrics:
    raise ValueError(f"Metric '{METRIC_KEY}' not found. Available: {list(metrics.keys())}")

rows.append(
    {
        "dataset": dataset_path.name,
        "method": METHOD_NAME,
        METRIC_KEY: float(metrics[METRIC_KEY]),
    }
)

results_df = pd.DataFrame(rows)
display(results_df)

Shape of scores: (4031,), Shape of labels: (4031,)
Shape of scores after conversion: (4031,)
Shape of scores after ravel: (4031,)


,dataset,method,AUC-ROC
0,001_NAB_id_1_Facility_tr_1007_1st_2014.csv,Sub_PCA,0.505614


In [12]:
print(f"Mean {METRIC_KEY}: {results_df[METRIC_KEY].mean():.4f}")

Mean AUC-ROC: 0.5056


In [16]:
# Run methods manually

# def run_Sub_PCA(data, periodicity=1, n_components=None, n_jobs=1):
#     from .models.PCA import PCA
#     slidingWindow = find_length_rank(data, rank=periodicity)
#     clf = PCA(slidingWindow = slidingWindow, n_components=n_components)
#     clf.fit(data)
#     score = clf.decision_scores_
#     return score.ravel()

sliding_window = 10
# what does clf stand for? classifier?
clf = PCA(slidingWindow=sliding_window, n_components=None)
clf.fit(data)
dir(clf)
scores = clf.decision_scores_
print(f"Shape of scores: {scores.shape}, Shape of labels: {labels.shape}")
scores = scores.ravel()
print(f"Shape of scores after ravel: {scores.shape}")
n = min(len(scores), len(labels))
scores = scores[:n]
labels = labels[:n]
metrics = get_metrics(scores, labels)
print(f"Metrics: {metrics}")


Shape of scores: (4031,), Shape of labels: (4031,)
Shape of scores after ravel: (4031,)
Metrics: {'AUC-PR': 0.2186001043543469, 'AUC-ROC': 0.5202824699759048, 'VUS-PR': 0.24298705247284808, 'VUS-ROC': 0.6210299229902944, 'Standard-F1': 0.22033501093414734, 'PA-F1': 1.0, 'Event-based-F1': 0.9999999999999996, 'R-based-F1': 0.44722096328952254, 'Affiliation-F': 0.976061577152747}
